In [1]:
# import pandas and RDKit modules to complete the first part of the workflow
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import AllChem

Experimental Hammett constants were taken from Hansch, Leo, and Taft's 1991 Chemical Reviews paper, cited by this work, from Table 1. Charged, ambiguous, and metal substituents were manually removed. The remaining substiutents were converted to SMILES strings.

In [ ]:
experimental_dataset = 'Chem_Rev_smiles_and_values.csv'

In [3]:
# read hammett paramaters and substituents from csv file
df = pd.read_csv(experimental_dataset)

In [ ]:
# create a list of RDKit mol objects from the substituent SMILES strings
smis = df['Substituent']
mol_objects = [Chem.MolFromSmiles(smi) for smi in smis]
df['mols'] = mol_objects

In [ ]:
# Filter out None values from mol_objects
valid_mol_objects = [mol for mol in mol_objects if mol is not None]

# use RDKit to calculate the number of rotatable bonds for each valid mol object
rot_bonds = [rdMolDescriptors.CalcNumRotatableBonds(mol) for mol in valid_mol_objects]

# Add number of rotatable bonds to the dataframe
df_valid = df[df['mols'].notna()].copy()
df_valid['rot_bonds'] = rot_bonds

In [ ]:
# In order to reduce the number of conformers for each substituent, filter out those with more than 2 rotatable bonds
df_filtered = df_valid[df_valid['rot_bonds'] < 3]

In [ ]:
# OPTIONAL – visualize the filtered substituents
Draw.MolsToGridImage(
    df_filtered['mols'], 
    molsPerRow=10, 
    subImgSize=(200, 200),
    maxMols = len(df_filtered['mols'])
)

In [ ]:
# define benzonitrile to react with the substituents
benzonitrile = Chem.MolFromSmiles('N#CC1=CC=CC=C1')

# Define the SMARTS reaction to make the para-substituted benzonitriles. The substituents were attached to At as a dummy atom. 
para_rxn = AllChem.ReactionFromSmarts('[N:1]#[C:2]-[c,C:3]1[c,C:4][c,C:5][c,C:6][c,C:7][c,C:8]1.[At]-[*:9]>>[N:1]#[C:2]-[c:3]1[c:4][c:5][c:6](-[*:9])[c:7][c:8]1')
para_rxn

In [ ]:
# Define the SMARTS reaction to make the meta-substituted benzonitriles.
meta_rxn = AllChem.ReactionFromSmarts('[N:1]#[C:2]-[c,C:3]1[c,C:4][c,C:5][c,C:6][c,C:7][c,C:8]1.[At]-[*:9]>>[N:1]#[C:2]-[c:3]1[c:4][c:5](-[*:9])[c:6][c:7][c:8]1')
meta_rxn

In [ ]:
# Run the SMARTS reaction to create the para- and meta-substituted benzonitriles for all of the substituents, then convert the products into SMILES strings.
para_products = []
for subst in df_filtered['mols']:
    print(f"Reacting benzonitrile with {subst}")
    result = para_rxn.RunReactants((benzonitrile, subst))
    print(f"Result: {result}")
    if result and result[0]:  # Check if there is at least one product
        para_products.append(result[0][0])
    else:
        para_products.append(None)  

para_smis = [Chem.MolToSmiles(mol) if mol is not None else '' for mol in para_products]
df_filtered['para_nitrile'] = para_smis

meta_products = []
for subst in df_filtered['mols']:
    print(f"Reacting benzonitrile with {subst}")
    result = meta_rxn.RunReactants((benzonitrile, subst))
    print(f"Result: {result}")
    if result and result[0]:  # Check if there is at least one product
        meta_products.append(result[0][0])
    else:
        meta_products.append(None)  

meta_smis = [Chem.MolToSmiles(mol) if mol is not None else '' for mol in meta_products]
df_filtered['meta_nitrile'] = meta_smis
df_filtered = df_filtered.drop(columns=['meta_cyano'])

In [ ]:
# create unique ids for para- and meta-substituted benzonitriles
para_nitrile_names = ['para_nitrile_{:03d}'.format(i) for i in range(len(para_smis))]
meta_nitrile_names = ['meta_nitrile_{:03d}'.format(i) for i in range(len(meta_smis))]
df_filtered['para_nitrile_id'] = para_nitrile_names
df_filtered['meta_nitrile_id'] = meta_nitrile_names

In [ ]:
# Save the filtered DataFrame with para- and meta-substituted benzonitriles to CSV files
df_filtered.drop(columns=['mols']).to_csv('nitriles.csv')
df_filtered[['para_nitrile_id', 'para_nitrile']].to_csv('para_nitriles.csv')
df_filtered[['meta_nitrile_id', 'meta_nitrile']].to_csv('meta_nitriles.csv')

para=pd.read_csv('para_nitriles.csv')
# rename columns so the csvs can be used to generate 3D conformers in the next step of the workflow
para = para.rename(columns={'para_nitrile_id': 'code_name', 'para_nitrile': 'SMILES'})
para.to_csv('para_nitriles.csv', index=False)

meta=pd.read_csv('meta_nitriles.csv')
meta = meta.rename(columns={'meta_nitrile_id': 'code_name', 'meta_nitrile': 'SMILES'})
meta.to_csv('meta_nitriles.csv', index=False)

In [ ]:
# Create a new conda environment (optional but recommended) and install Automated Quantum Mechanical Environments (AQME), a computational chemistry python package 
# RUN THIS CELL ONLY ONCE
! conda create -n aqme python=3.10
! conda activate aqme
! pip install aqme
! conda install -y -c conda-forge openbabel=3.1.1

In [ ]:
# use AQME to generate 3D conformers from the SMILES of the para-substituted benzonitriles
! python -m aqme --csearch --program rdkit --input para_nitriles.csv 

In [ ]:
# use AQME to generate 3D conformers from the SMILES of the meta-substituted benzonitriles
! python -m aqme --csearch --program rdkit --input meta_nitriles.csv

In [ ]:
# use AQME to prepare the conformers for quantum mechanical optimization, using the benchmarked level of theory
# Here we use Gaussian to perform the calculations, but the program can be changed based on your needs
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2SVP emp=GD3BJ opt freq" --files "CSEARCH/*.sdf"
# OPTIONAL – rename the QCALC directory
! mv QCALC optimization_inputs

Manually inspect the quantum mechanical input files to ensure accurate creation/change number of processors, memory, or
anything else you might want to change. Submit the calculations using the selected computational chemistry program. Once the 
calculations are complete and placed in an "outputs" directory, you can proceed to the next step of the workflow.

In [ ]:
# use AQME to check the optimization output files for successful completion
# AQME will generate new input files for those calculations that did not terminate successfully/did not optimize to a minimum. Resubmit these calculations.
! python -m aqme --qcorr --program gaussian --files "optimization_outputs/*.log"

Two files (para_nitrile_03_rdkit_conf_1 and para_nitrile_197_rdkit_conf_1) 
have small imaginary frequencies (< 8 cm^-1) that correspond to vibrations similar 
to a methyl group rotation. Because these imaginary frequencies are small, we are 
keeping these files in the dataset and proceeding.

In [ ]:
# Use AQME to generate single point calculation inputs for the successfully optimized conformers, using the benchmarked level of theory
# pop=(Hirshfeld) will compute both Hirshfeld and CM5 charges that will be used for predictions
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2QZVPP emp=GD3BJ pop=(Hirshfeld)" --files "optimization_outputs/success/*.log"
# OPTIONAL – rename the directory containing the input files
! mv QCALC single_point_inputs

In [ ]:
# Use AQME to check the single point output files for successful completion
! python -m aqme --qcorr --program gaussian --files "single_point_outputs/*.log"

The next steps require molDscript, a python package developed in the Paton Lab at CSU. Visit https://github.com/patonlab/molDscript to install molDscript and the associated packages.


In [ ]:
# OPTIONAL – rename the directories for the molDscript input and separate meta and para – substituted benzonitriles
! mkdir meta_opt 
! cp optimization_outputs/success/meta_*.log meta_opt
! mkdir meta_SP 
! cp single_point_outputs/success/SP_calcs/meta_*.log meta_SP

In [ ]:
# substructure obtained using University of Hamburg SMARTSeditor

# use molDscript to parse atomic charges from the meta single point calculations
! python -m moldscript --opt meta_opt --spc meta_SP --charges meta_SP --substructure 'cC#N' --boltz --min_max --lowe
! mkdir meta_charges
! mv *_level.csv meta_charges
! mv MOLDSCRIPT*.dat meta_charges

In [ ]:
# Organize the molDscript output data. The atom_level csv contains the atomic charge information.
meta_dir = 'meta_charges/'
df_meta = pd.read_csv(meta_dir + 'lowest_energy_atom_level.csv')

# separate N from other atom types
df_N_meta = df_meta[df_meta['atom_type'] == 'N']

# remove the 'rdkit' suffix
df_N_meta['filename'] = df_N_meta['filename'].apply(lambda x: '_'.join([x.split('_')[0], x.split('_')[1], x.split('_')[2].zfill(3)]))

# reorder the dataframe
df_N_meta = df_N_meta.sort_values(by='filename')

# rename the columns to match the expected format
df_N_meta = df_N_meta.rename(columns={'filename': 'meta_nitrile_id'})

# read in the experimental values and add the experimental Hammett parameter to the dataframe
exp_values = pd.read_csv('nitriles.csv')
exp_values = exp_values[['meta_nitrile_id', 'sigma_m']]
compare_meta = pd.merge(df_N_meta, exp_values, on='meta_nitrile_id')
compare_meta.to_csv('meta_charges.csv', index=False)

In [ ]:
# identify any benzonitriles with more than one N atom (incorrectly parsed charges)
meta_charges = pd.read_csv('meta_charges.csv')
repeats = meta_charges[meta_charges.duplicated('meta_nitrile_id', keep=False)]
repeats_df = repeats.copy()
print(repeats_df)

In [ ]:
# manually examine the repeats to identify the correct nitrogen atom
# remove incorrectly parsed N atoms from the dataset
meta_charges = pd.read_csv('meta_charges.csv')
meta_charges = meta_charges[~((meta_charges['meta_nitrile_id'] == 'meta_nitrile_072') & (meta_charges['atom_index'] != 10))]
meta_charges = meta_charges[~((meta_charges['meta_nitrile_id'] == 'meta_nitrile_375') & (meta_charges['atom_index'] != 16))]
# re-save the csv with the correct data
meta_charges.to_csv('meta_charges.csv', index=False)

In [ ]:
# Repeat the above but with the para-substituted benzonitriles
! python -m aqme --qcorr --program gaussian --files "single_point_outputs/*.log"

In [ ]:
! mkdir para_opt 
! cp optimization_outputs/success/para_*.log para_opt
! mkdir para_SP 
! cp single_point_outputs/success/SP_calcs/para_*.log para_SP

In [ ]:
! python -m moldscript --opt para_opt --spc para_SP --charges para_SP --substructure 'cC#N' --boltz --min_max --lowe
! mkdir para_charges
! mv *_level.csv para_charges
! mv MOLDSCRIPT*.dat para_charges

In [ ]:
para_dir = 'para_charges/'
df_para = pd.read_csv(para_dir + 'lowest_energy_atom_level.csv')
df_N_para = df_para[df_para['atom_type'] == 'N']
df_N_para['filename'] = df_N_para['filename'].apply(lambda x: '_'.join([x.split('_')[0], x.split('_')[1], x.split('_')[2].zfill(3)]))
df_N_para = df_N_para.sort_values(by='filename')
df_N_para = df_N_para.rename(columns={'filename': 'para_nitrile_id'})
exp_values = pd.read_csv('nitriles.csv')
exp_values = exp_values[['para_nitrile_id', 'sigma_p']]
compare_para = pd.merge(df_N_para, exp_values, on='para_nitrile_id')
compare_para.to_csv('para_charges.csv', index=False)

In [ ]:
para_charges = pd.read_csv('para_charges.csv')
repeats = para_charges[para_charges.duplicated('para_nitrile_id', keep=False)]
repeats_df = repeats.copy()
print(repeats_df)

In [ ]:
para_charges = pd.read_csv('para_charges.csv')
para_charges = para_charges[~((para_charges['para_nitrile_id'] == 'para_nitrile_031') & (para_charges['atom_index'] != 1))]
para_charges = para_charges[~((para_charges['para_nitrile_id'] == 'para_nitrile_070') & (para_charges['atom_index'] != 1))]
para_charges = para_charges[~((para_charges['para_nitrile_id'] == 'para_nitrile_072') & (para_charges['atom_index'] != 9))]
para_charges = para_charges[~((para_charges['para_nitrile_id'] == 'para_nitrile_375') & (para_charges['atom_index'] != 15))]
para_charges.to_csv('para_charges.csv', index=False)

In [ ]:
# Combine the meta and para substituent charge data into a single CSV file
metadata = pd.read_csv('meta_charges.csv')
paradata = pd.read_csv('para_charges.csv')
metadata = metadata.rename(columns={'sigma_m': 'hammett_parameter'})
paradata = paradata.rename(columns={'sigma_p': 'hammett_parameter'})
metadata['nitrile_id'] = metadata['meta_nitrile_id']
paradata['nitrile_id'] = paradata['para_nitrile_id']
metadata = metadata.drop(columns=['meta_nitrile_id'])
paradata = paradata.drop(columns=['para_nitrile_id'])
combined = pd.concat([metadata, paradata], ignore_index=True)
combined.to_csv('combined_charges.csv', index=False)